# 04 `accumulation_steps`：两个小批次，更新一次

上一课的循环是：

```text
zero_grad → loss → backward → step
```

这一课加入一个工程问题：显存不够时，能不能让多个小批次合起来，像一个更大的批次那样更新？答案是梯度累积。

## 知识点标注

- 行业状态：常见工程实践。大模型训练经常用它模拟更大的有效 batch size。
- 现实用途：显存只能装小批次时，多个小批次先分别计算梯度，再统一更新参数。
- 学习深度：能解释‘什么时候累积、什么时候 step、为什么 loss 要除以累积步数’。
- 本课类型：概念演示。只创建一个参数，不使用 GPU，不训练 MokioMind。

## 先看两种节奏

普通训练，每个小批次都更新：

```text
小批次 A：backward → step
小批次 B：backward → step
```

梯度累积，两个小批次才更新一次：

```text
小批次 A：backward        （先不 step）
小批次 B：backward        （梯度继续累积）
然后：step → zero_grad
```

`accumulation_steps=2` 的意思是：两个小批次合并成一次参数更新。它不是让一个小批次的形状变成两倍。

In [1]:
import torch

target_a = 3.0
target_b = 5.0
lr = 0.1
accumulation_steps = 2

print('两个小批次的目标：', target_a, '和', target_b)
print('累积步数：', accumulation_steps)

两个小批次的目标： 3.0 和 5.0
累积步数： 2


### 先预测

从 `w=1.0` 开始：

```text
样本 A：loss=(1-3)²=4，梯度应该是 -4
样本 B：loss=(1-5)²=16，梯度应该是 -8
```

如果两个 loss 都先除以 2，再依次 `backward()`，第二次 backward 后的总梯度应该是多少？

提示：这里 PyTorch 默认是累加，不会覆盖第一次梯度。

In [2]:
w = torch.tensor([1.0], requires_grad=True)
optimizer = torch.optim.SGD([w], lr=lr)

optimizer.zero_grad()
loss_a = (w - target_a) ** 2 / accumulation_steps
loss_a.backward()
print('A backward 后：w =', w.item(), 'grad =', w.grad.item())

loss_b = (w - target_b) ** 2 / accumulation_steps
loss_b.backward()
print('B backward 后：w =', w.item(), 'grad =', w.grad.item())
print('预期：(-4 / 2) + (-8 / 2) = -6')

A backward 后：w = 1.0 grad = -2.0
B backward 后：w = 1.0 grad = -6.0
预期：(-4 / 2) + (-8 / 2) = -6


## 什么时候参数才变化？

刚才两次 `backward()` 都只改了 `w.grad`，`w` 仍然是 `1.0`。现在才执行一次 `step()`：

```text
平均梯度 = -6
新 w = 1.0 - 0.1 × (-6) = 1.6
```

In [3]:
print('step 前：w =', w.item(), 'grad =', w.grad.item())
optimizer.step()
print('step 后：w =', w.item())
optimizer.zero_grad()
print('zero_grad 后：w.grad =', w.grad)

step 前：w = 1.0 grad = -6.0
step 后：w = 1.600000023841858
zero_grad 后：w.grad = None


## 为什么每个 loss 要除以 `accumulation_steps`？

如果不除以 2：

```text
总梯度 = -4 + (-8) = -12
更新量 = 0.1 × 12 = 1.2
w 会从 1.0 直接走到 2.2
```

除以 2 后：

```text
总梯度 = (-4 / 2) + (-8 / 2) = -6
这等于两个小批次梯度的平均值。
```

所以除法的作用不是改变正确答案，而是让一次更新的规模保持在‘大批次平均梯度’的水平。

In [4]:
w_sum = torch.tensor([1.0], requires_grad=True)
optimizer_sum = torch.optim.SGD([w_sum], lr=lr)

optimizer_sum.zero_grad()
((w_sum - target_a) ** 2).backward()
((w_sum - target_b) ** 2).backward()
print('不除以 2 的累计梯度：', w_sum.grad.item())
optimizer_sum.step()
print('不除以 2 的更新结果：w =', w_sum.item())
print('对照：除以 2 时 w = 1.6；不除时步子变成两倍')

不除以 2 的累计梯度： -12.0
不除以 2 的更新结果：w = 2.200000047683716
对照：除以 2 时 w = 1.6；不除时步子变成两倍


## 对应 MokioMind 代码

项目训练循环的关键部分是：

```python
loss = res.loss + res.aux_loss
loss = loss / args.accumulation_steps
scaler.scale(loss).backward()

if step % args.accumulation_steps == 0:
    scaler.step(optimizer)
    optimizer.zero_grad(set_to_none=True)
```

如果 `accumulation_steps=4`，就会连续处理 4 个小批次、累积 4 次梯度，之后只更新一次。`loss` 除以 4 后，累计结果近似 4 个小批次梯度的平均值。

## 验收题

1. `accumulation_steps=2` 时，第一次 `backward()` 后参数会更新吗？
2. 两次 `backward()` 之间为什么不能 `zero_grad()`？
3. 为什么每个 loss 要除以 2？不除会发生什么？
4. 项目里真正执行参数更新的是哪一行？